# Preprocesamiento y diccionario de Online Retail II — v04
Continuación del cierre v03. Este cuaderno corto permite trabajar sin volver a leer el Excel.

Entrada: `base_limpia_v03.csv.gz`, disponible dentro de las evidencias del cierre. Extraer el ZIP; el CSV puede permanecer comprimido en .gz. También puede usarse la variable `base_limpia` si se copian estas celdas al final del cuaderno v03 en la misma sesión.

No construye aún RFM, etiquetas ni modelos. No se imputan ni escalan datos. Los identificadores se usan para agrupar, no como predictores numéricos.

## Carga de la base limpia
**Propósito:** continuar exactamente desde la base del cierre.
**Operaciones:** reutilizar base_limpia o cargar el CSV, solicitándolo en Colab si falta. Verificar 805.935 filas y 5.875 clientes.
**Resultado:** una copia en memoria; el archivo de entrada no se modifica.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import zipfile
if 'base_limpia' in globals():
    entrada = base_limpia.copy(deep=True)
else:
    opciones = [Path('base_limpia_v03.csv.gz'), Path('salidas_limpieza_v03/base_limpia_v03.csv.gz'), Path('evidencias/base_limpia_v03.csv.gz')]
    archivo = next((p for p in opciones if p.is_file()), None)
    if archivo is None:
        from google.colab import files
        subidos = files.upload()
        candidatos = [Path(p) for p in subidos if p.endswith('.csv.gz')]
        if len(candidatos) != 1:
            raise ValueError('Carga solamente base_limpia_v03.csv.gz.')
        archivo = candidatos[0]
        del subidos
    entrada = pd.read_csv(archivo, dtype={'Invoice': 'string', 'StockCode': 'string'})
assert len(entrada) == 805935
assert entrada['Customer ID'].nunique() == 5875
print('Entrada:', len(entrada), 'líneas y', entrada['Customer ID'].nunique(), 'clientes.')


## Estandarización y controles
**Propósito:** asignar a cada variable un tipo compatible con su significado.
**Operaciones:** convertir Customer ID de número entero representado como decimal a texto; conservar los códigos Invoice y StockCode como texto; convertir fechas sin inventar zona horaria; verificar cantidades enteras y valores monetarios finitos. No se eliminan filas ni se cambia su orden.
**Resultado esperado:** clientes como `14096`, sin `.0`; fechas utilizables en ventanas temporales y cantidades/importes numéricos.
**Interpretación:** convertir tipos no equivale a escalar. Las comprobaciones validan conservación por fila y que no se fusionen identificadores. Los documentos incluyen compras y cancelaciones, por lo que su total no se interpreta como número de compras.


In [ ]:
preprocesada = entrada.copy(deep=True)
ids = pd.to_numeric(entrada['Customer ID'], errors='raise')
assert ids.notna().all() and np.isfinite(ids).all()
assert ids.gt(0).all() and ids.eq(np.floor(ids)).all()
preprocesada['Customer ID'] = ids.astype('Int64').astype('string')
assert preprocesada['Customer ID'].nunique() == entrada['Customer ID'].nunique()
assert np.array_equal(pd.to_numeric(preprocesada['Customer ID']).to_numpy(), ids.to_numpy())
for c in ['Invoice', 'StockCode', 'Description', 'Country', 'hoja_origen', 'concepto_limpieza', 'motivo_limpieza']:
    preprocesada[c] = entrada[c].astype('string')
    assert preprocesada[c].equals(entrada[c].astype('string'))
preprocesada['InvoiceDate'] = pd.to_datetime(entrada['InvoiceDate'], errors='raise')
assert preprocesada.InvoiceDate.notna().all()
for c in ['Quantity', 'fila_excel']:
    v = pd.to_numeric(entrada[c], errors='raise')
    assert np.isfinite(v).all() and v.eq(np.floor(v)).all()
    preprocesada[c] = v.astype('int64')
for c in ['Price', 'Importe_linea']:
    preprocesada[c] = pd.to_numeric(entrada[c], errors='raise').astype('float64')
    assert np.isfinite(preprocesada[c]).all()
for c in ['Quantity', 'Price', 'Importe_linea', 'fila_excel']:
    assert np.array_equal(preprocesada[c].to_numpy(), entrada[c].to_numpy())
assert preprocesada.index.equals(entrada.index)
assert len(preprocesada) == len(entrada)
assert np.allclose(preprocesada.Quantity * preprocesada.Price, preprocesada.Importe_linea, rtol=1e-12, atol=1e-9)
assert not preprocesada.duplicated(['hoja_origen', 'fila_excel']).any()
assert preprocesada.groupby('Invoice')['Customer ID'].nunique().max() == 1
control_tipos = pd.DataFrame({'variable': entrada.columns, 'tipo_entrada': entrada.dtypes.astype(str).values,
    'tipo_salida': preprocesada.dtypes.astype(str).values, 'nulos_salida': preprocesada.isna().sum().values})
control_conservacion = pd.DataFrame([
    ['Filas', len(entrada), len(preprocesada)],
    ['Clientes distintos', entrada['Customer ID'].nunique(), preprocesada['Customer ID'].nunique()],
    ['Documentos distintos (no compras)', entrada.Invoice.nunique(), preprocesada.Invoice.nunique()],
    ['Importe firmado GBP', entrada.Importe_linea.sum(), preprocesada.Importe_linea.sum()],
    ['Líneas negativas', int(entrada.Importe_linea.lt(0).sum()), int(preprocesada.Importe_linea.lt(0).sum())]
], columns=['control','antes','despues'])
display(control_tipos)
with pd.option_context('display.float_format', '{:,.3f}'.format):
    display(control_conservacion)
print('Controles aprobados. No se modificaron filas, cantidades ni importes.')


## Diccionario de la base transaccional y exportación
**Propósito:** documentar descripción, tipo, unidad y función de cada columna existente.
**Operaciones:** generar una tabla editable CSV y guardar base, controles y bitácora en un ZIP.
**Interpretación:** este diccionario describe las 13 columnas actuales. Las futuras variables cliente-corte requieren un diccionario adicional una vez definidas sus ventanas y fórmulas. No confundir las columnas de auditoría con variables predictoras.

Al recargar CSV, declarar tipos de identificadores y convertir InvoiceDate: CSV no conserva tipos de pandas.


In [ ]:
filas_diccionario = [['Invoice', 'Identificador del documento; C indica cancelación, no motivo específico.', 'Nominal', 'Sin unidad', 'Agrupación documental; no predictor directo'], ['StockCode', 'Código del producto o concepto; D identifica descuento.', 'Nominal', 'Sin unidad', 'Diversidad futura de mercancía; excluir D del conteo'], ['Description', 'Descripción registrada; puede variar para un mismo código.', 'Texto', 'Sin unidad', 'Interpretación y auditoría'], ['Quantity', 'Cantidad con signo; en descuentos no implica unidades físicas.', 'Cuantitativa discreta', 'Unidades registradas', 'Cálculo de importe'], ['InvoiceDate', 'Fecha y hora registrada; zona horaria no documentada.', 'Temporal', 'Fecha y hora', 'Ventanas y disponibilidad al corte'], ['Price', 'Precio unitario positivo en la base limpia.', 'Cuantitativa', 'GBP por unidad registrada', 'Cálculo de importe'], ['Customer ID', 'Código nominal del cliente, representado sin decimales.', 'Nominal', 'Sin unidad', 'Agrupación cliente-corte; no predictor directo'], ['Country', 'País asociado al cliente en el registro.', 'Nominal', 'Sin unidad', 'Descripción geográfica; no selección como predictor aún'], ['hoja_origen', 'Hoja del Excel de procedencia.', 'Nominal', 'Sin unidad', 'Trazabilidad'], ['fila_excel', 'Número de fila de procedencia, considerando encabezado.', 'Identificador de posición', 'Fila', 'Trazabilidad; con hoja identifica la línea'], ['Importe_linea', 'Quantity × Price, preservando el signo y precisión.', 'Cuantitativa', 'GBP', 'Agregación de ventas netas; no es aún objetivo a 90 días'], ['concepto_limpieza', 'Mercancía según definición operativa o descuento.', 'Nominal', 'Sin unidad', 'Separar importe de actividad de compra'], ['motivo_limpieza', 'Conservar en esta base; razones de separación en auditoría.', 'Nominal', 'Sin unidad', 'Control de limpieza; no predictor']]
diccionario = pd.DataFrame(filas_diccionario, columns=['variable','descripcion','naturaleza','unidad','funcion'])
diccionario['tipo_pandas'] = diccionario.variable.map(preprocesada.dtypes.astype(str))
assert set(diccionario.variable) == set(preprocesada.columns)
display(diccionario)
salida = Path('salidas_preprocesamiento_v04')
salida.mkdir(exist_ok=True)
preprocesada.to_csv(salida / 'base_preprocesada_v04.csv.gz', index=False, compression='gzip')
diccionario.to_csv(salida / 'diccionario_transaccional_v04.csv', index=False, encoding='utf-8-sig')
control_tipos.to_csv(salida / 'control_tipos.csv', index=False, encoding='utf-8-sig')
control_conservacion.to_csv(salida / 'control_conservacion.csv', index=False, encoding='utf-8-sig')
bitacora = {'version':'0.4', 'fecha':'2026-10-23', 'decision':'Identificadores como texto; cantidades enteras; fechas sin zona horaria inferida; importes sin redondeo',
 'estado':'Aplicada y verificada', 'filas':len(preprocesada), 'clientes':int(preprocesada['Customer ID'].nunique()),
 'diccionario':'13 variables transaccionales; no matriz predictiva',
 'pendientes':['Ventanas y elegibilidad temporal', 'Indicadores por cliente', 'Objetivo a 90 días', 'EDA por cliente'],
 'versiones':{'pandas':pd.__version__,'numpy':np.__version__}}
(salida / 'bitacora_v04.json').write_text(json.dumps(bitacora, ensure_ascii=False, indent=2),encoding='utf-8')
paquete = Path('Evidencias_preprocesamiento_v04.zip')
with zipfile.ZipFile(paquete,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(salida.iterdir()): z.write(f, arcname=f.name)
print('Evidencias:', paquete.resolve())
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(paquete))
